In [1]:
import pandas as pd

df = pd.read_csv("cfb_qb_stats.csv")

if "Rate.1" in df.columns:
    df = df.drop(columns=["Rate.1"])

if "-9999" in df.columns:
    df = df.drop(columns=["-9999"])

df = df.loc[:, ~df.columns.str.contains("Unnamed")]

# columns getting qb prefix
qb_stat_cols = [
    "Cmp", "Att", "Inc", "Cmp%", "Yds", "TD", "Int",
    "TD%", "Int%", "Y/A", "AY/A", "Y/C", "Y/G"
]

# renaming only those columns
rename_dict = {col: f"qb_{col}" for col in qb_stat_cols if col in df.columns}
df = df.rename(columns=rename_dict)

df = df.reset_index(drop=True)

df.to_csv("cfb_qb_stats_clean.csv", index=False)

In [7]:
df = pd.read_csv("cfb_rb_stats.csv")

df = df.loc[:, ~df.columns.str.contains("Unnamed")]
df = df.drop(columns=["-9999"], errors="ignore")

# rename rushing + receiving stats
rename_dict = {

    # rushing
    "Att": "rb_rush_Att",
    "Yds": "rb_rush_Yds",
    "Y/A": "rb_rush_Y_per_Att",
    "TD": "rb_rush_TD",

    # receiving
    "Rec": "rb_rec_Rec",
    "Yds.1": "rb_rec_Yds",
    "Y/R": "rb_rec_Y_per_Rec",
    "TD.1": "rb_rec_TD",
}

df = df.rename(columns=rename_dict)

# explicitly assign correct Y/G columns
df["rb_rush_Y_per_Game"] = df["Y/G.2"]   # rushing yards per game
df["rb_rec_Y_per_Game"] = df["Y/G.1"]    # receiving yards per game

# remove all original duplicate Y/G columns
df = df.drop(columns=[c for c in df.columns if c.startswith("Y/G")])

df = df.reset_index(drop=True)

df.to_csv("cfb_rb_stats_clean.csv", index=False)

In [9]:
df = pd.read_csv("cfb_wrte_stats.csv")

if "-9999" in df.columns:
    df = df.drop(columns=["-9999"])
df = df.loc[:, ~df.columns.str.contains("Unnamed")]

if "Y/G.1" in df.columns:
    df = df.drop(columns=["Y/G.1"])

receiving_cols = {
    "Rec": "rec_Rec",
    "Yds": "rec_Yds",
    "Y/R": "rec_Y/R",
    "TD": "rec_TD",
    "Y/G": "rec_Y/G"
}

rename_dict = {col: new_col for col, new_col in receiving_cols.items() if col in df.columns}
df = df.rename(columns=rename_dict)

df = df.reset_index(drop=True)

df.to_csv("cfb_wrte_stats_clean.csv", index=False)

In [10]:
df = pd.read_csv("cfb_ol_stats.csv")

df = df.loc[:, ~df.columns.str.contains("Unnamed")]  
if "-9999" in df.columns:
    df = df.drop(columns=["-9999"])
if "Player-additional" in df.columns:
    df = df.drop(columns=["Player-additional"])

cols_to_drop = []
if "G.1" in df.columns:
    cols_to_drop.append("G.1")
df = df.drop(columns=cols_to_drop)

df = df.reset_index(drop=True)

df.to_csv("cfb_ol_stats_clean.csv", index=False)

In [12]:
import numpy as np

df = pd.read_csv("cfb_dllb_stats.csv")

df = df.loc[:, ~df.columns.str.contains("Unnamed")]
df = df.drop(columns=["-9999"], errors="ignore")

df = df.reset_index(drop=True)

# drop redundant sack column
redundant_sk_col = df.columns[2]
df = df.drop(columns=[redundant_sk_col])

if "Sk.1" in df.columns:
    df = df.rename(columns={"Sk.1": "Sk"})

df["Sk"] = pd.to_numeric(df["Sk"], errors="coerce")

# avoid divide-by-zero
df["G"] = df["G"].replace(0, np.nan)

# sacks per game
df["Sk/G"] = df["Sk"] / df["G"]

# converting other defensive stats to per game
cols_to_convert = ["Solo", "Ast", "Comb", "TFL", "Int", "IntTD", "PD"]

for col in cols_to_convert:
    if col in df.columns:
        df[f"{col}/G"] = df[col] / df["G"]

# drop unnecessary yard columns
for col in ["Yds", "Yds.1"]:
    if col in df.columns:
        df = df.drop(columns=[col])

df.to_csv("cfb_dllb_stats_clean.csv", index=False)

In [63]:
df = pd.read_csv("college_db_stats1.csv")

df = df.loc[:, ~df.columns.str.contains("Unnamed")]
df = df.drop(columns=["-9999"], errors="ignore")

redundant_int_col = df.columns[2]  
df = df.drop(columns=[redundant_int_col])

if "Int.1" in df.columns:
    df = df.rename(columns={"Int.1": "Int"})

columns_to_drop = ["Yds", "Yds.1", "Yds.2", "TotOff", "Touch"]
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

df = df.reset_index(drop=True)

df.to_csv("college_db_stats_clean1.csv", index=False)

In [17]:
df = pd.read_csv("cfb_db_stats.csv")

df = df.loc[:, ~df.columns.str.contains("Unnamed")]
df = df.drop(columns=["-9999"], errors="ignore")

# drop redundant interception column
redundant_int_col = df.columns[2]
df = df.drop(columns=[redundant_int_col])

# rename interception column
if "Int.1" in df.columns:
    df = df.rename(columns={"Int.1": "Int"})

# rename interception return yards
if "Yds" in df.columns:
    df = df.rename(columns={"Yds": "db_ret_Yds"})

# convert stats to numeric
df["Int"] = pd.to_numeric(df["Int"], errors="coerce")
df["PD"] = pd.to_numeric(df["PD"], errors="coerce")
df["G"] = pd.to_numeric(df["G"], errors="coerce")

# convert PD = 0 to NaN
df["PD"] = df["PD"].replace(0, np.nan)

# avoid divide-by-zero
df["G"] = df["G"].replace(0, np.nan)

# per-game stats
df["Int/G"] = df["Int"] / df["G"]
df["PD/G"] = df["PD"] / df["G"]

df = df.reset_index(drop=True)

df.to_csv("cfb_db_stats_clean.csv", index=False)

In [18]:
qb = pd.read_csv("cfb_qb_stats_clean.csv")
rb = pd.read_csv("cfb_rb_stats_clean.csv")
wrte = pd.read_csv("cfb_wrte_stats_clean.csv")
ol = pd.read_csv("cfb_ol_stats_clean.csv")
dllb = pd.read_csv("cfb_dllb_stats_clean.csv")
db = pd.read_csv("cfb_db_stats_clean.csv")

dfs = [qb, rb, wrte, ol, dllb, db]

combined_df = pd.concat(dfs, ignore_index=True, sort=False)

if "Rk" in combined_df.columns:
    combined_df = combined_df.drop(columns=["Rk"])

# sorting by draft yr
combined_df = combined_df.reset_index(drop=True)
combined_df = combined_df.sort_values(by=["Draft Year", "Round", "Pick"], ignore_index=True)

combined_df.to_csv("cfb_all_positions_clean.csv", index=False)

In [20]:
import cfbd
import pandas as pd
import time


configuration = cfbd.Configuration(
    access_token="II7zE93U5v+z5UPByQ1arKsJsU14FUpEkmqMkpKRyzzgrE/kF3a1jtbKNcFEO9P/"
)

START_YEAR = 1995  # to be safe cus want to get all data for players drafted in 2000 and some may have had long college careers
END_YEAR = 2020

# getting team SP ratings
sp_rows = []

with cfbd.ApiClient(configuration) as api_client:
    ratings_api = cfbd.RatingsApi(api_client)
    for year in range(START_YEAR, END_YEAR + 1):
        print(f"Pulling SP+ for {year}...")
        try:
            ratings = ratings_api.get_sp(year=year)
            for team in ratings:
                sp_rows.append({
                    "year": year,
                    "team": team.team,
                    "sp_rating": team.rating
                })
        except Exception as e:
            print(f"Error pulling SP+ for {year}: {e}")
        time.sleep(0.3)

sp_df = pd.DataFrame(sp_rows)

# getting player info
season_rows = []

with cfbd.ApiClient(configuration) as api_client:
    stats_api = cfbd.StatsApi(api_client)
    for year in range(START_YEAR, END_YEAR + 1):
        print(f"Pulling player season stats for {year}...")
        try:
            stats = stats_api.get_player_season_stats(year=year)
            for entry in stats:
                if entry.team is None:
                    continue
                season_rows.append({
                    "player": entry.player,
                    "year": year,
                    "team": entry.team,
                    "college": getattr(entry, "school", None)
                })
        except Exception as e:
            print(f"Error pulling stats for {year}: {e}")
        time.sleep(0.3)

season_df = pd.DataFrame(season_rows)


merged = season_df.merge(
    sp_df,
    on=["year", "team"],
    how="left"
)

merged = merged.dropna(subset=["sp_rating"])


# avg SP for a player during college career
player_avg_sp = (
    merged
    .groupby("player")["sp_rating"]
    .mean()
    .reset_index()
    .rename(columns={"sp_rating": "avg_team_sp"})
)


college_df = pd.read_csv("cfb_all_positions_clean.csv")

# merging players' career avg SP with their college production data by draft year, round, and pick since unique
player_full = college_df.merge(
    player_avg_sp,
    left_on="Player",  
    right_on="player",
    how="left"
).drop(columns=["player"])

# sorting
player_full_sorted = player_full.sort_values(
    by=["Draft Year", "Round", "Pick"],
    ascending=[True, True, True]
).reset_index(drop=True)


# adding AV and other draft info
draft_info = pd.read_csv("draft_2000_2026.csv")

# columns we want from df
draft_subset = draft_info[["season", "round", "pick", "age", "dr_av", "w_av"]]


full_df = player_full_sorted.merge(
    draft_subset,
    left_on=["Draft Year", "Round", "Pick"],
    right_on=["season", "round", "pick"],
    how="left"
)

full_df = full_df.drop(columns=["season", "round", "pick"])

full_df = full_df.sort_values(
    by=["Draft Year", "Round", "Pick"],
    ascending=[True, True, True]
).reset_index(drop=True)

Pulling SP+ for 1995...
Error pulling SP+ for 1995: 1 validation error for TeamSP
rating
  none is not an allowed value (type=type_error.none.not_allowed)
Pulling SP+ for 1996...
Pulling SP+ for 1997...
Error pulling SP+ for 1997: 1 validation error for TeamSP
rating
  none is not an allowed value (type=type_error.none.not_allowed)
Pulling SP+ for 1998...
Pulling SP+ for 1999...
Pulling SP+ for 2000...
Pulling SP+ for 2001...
Error pulling SP+ for 2001: 1 validation error for TeamSPDefense
rating
  none is not an allowed value (type=type_error.none.not_allowed)
Pulling SP+ for 2002...
Pulling SP+ for 2003...
Pulling SP+ for 2004...
Error pulling SP+ for 2004: 1 validation error for TeamSPDefense
rating
  none is not an allowed value (type=type_error.none.not_allowed)
Pulling SP+ for 2005...
Pulling SP+ for 2006...
Pulling SP+ for 2007...
Pulling SP+ for 2008...
Pulling SP+ for 2009...
Pulling SP+ for 2010...
Pulling SP+ for 2011...
Pulling SP+ for 2012...
Pulling SP+ for 2013...
Pullin

In [21]:
full_df

,Player,Rate,Draft Team,Round,Pick,Draft Year,Draft College,From,To,G,...,Int,db_ret_Yds,IntTD,PD,Int/G,PD/G,avg_team_sp,age,dr_av,w_av
0,Courtney Brown,NaN,CLE,1,1,2000,Penn State,1999,1999,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,21.0,27.0
1,Lavar Arrington,NaN,WAS,1,2,2000,Penn State,1998,1999,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,45.0,46.0
2,Chris Samuels,NaN,WAS,1,3,2000,Alabama,1999,1999,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.0,63.0,63.0
3,Peter Warrick,NaN,CIN,1,4,2000,Florida State,1995,1999,54.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.0,25.0,27.0
4,Jamal Lewis,NaN,RAV,1,5,2000,Tennessee,1997,1999,27.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0,53.0,69.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4839,Samori Toure,NaN,GNB,7,258,2022,Nebraska,2021,2021,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0,2.0,2.0
4840,Nazeeh Johnson,NaN,KAN,7,259,2022,Marshall,2017,2021,56.0,...,7.0,63.0,1.0,19.0,0.125,0.339286,1.202000,24.0,3.0,3.0
4841,Alexander Horvath,NaN,SDG,7,260,2022,Purdue,2018,2021,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.770000,23.0,0.0,0.0
4842,AJ Arcuri,NaN,RAM,7,261,2022,Michigan State,2020,2021,20.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.0,1.0,1.0


In [22]:
full_df.to_csv("cfb_stats_with_avg_sp_and_av.csv", index=False)